Mistral Large 3 is Mistral’s first mixture-of-experts model since the seminal Mixtral series, and represents a substantial step forward in pretraining at Mistral. After post-training, the model achieves parity with the best instruction-tuned open-weight models on the market on general prompts, while also demonstrating image understanding and best-in-class performance on multilingual conversations (i.e., non-English/Chinese).

Mistral Large 3 debuts at #2 in the OSS non-reasoning models category (#6 amongst OSS models overall) on the <a href="https://lmarena.ai/leaderboard/text">LMArena leaderboard</a> and is released under the Apache 2.0 license.

In this notebook, we will show how some basics on how to work with Mistral Large 3 in Microsoft Foundry, with a focus on tool calling using MCP.

In [ ]:
import json
import requests
import os
from dotenv import load_dotenv
from typing import Any
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam


from IPython.display import Image, display  

In [15]:

load_dotenv()

AZURE_MISTRAL_MODEL_ENDPOINT = os.getenv("AZURE_MISTRAL_MODEL_ENDPOINT")
AZURE_MISTRAL_MODEL_KEY = os.getenv("AZURE_MISTRAL_MODEL_KEY")

AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
PROJECT_API_NAME = os.getenv("PROJECT_API_NAME")

AZURE_AI_DEPLOYMENT_NAME = os.getenv("AZURE_AI_DEPLOYMENT_NAME")

MCP_SERVER_URL = os.getenv("MCP_SERVER_URL")
MCP_CONNECTION_NAME= os.getenv("MCP_CONNECTION_NAME")



REQUEST_HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {AZURE_MISTRAL_MODEL_KEY}",
}

Local Tool Call

In [12]:
def documentPayload_tool(question: str) -> dict[str, Any]:
  ## This is a sample payload for sending a document to the Mistral Document AI API. The "content" field contains a list of message parts, which can include text and images. In this example, we are sending a text prompt along with an image encoded in base64 format. The model specified in the payload will process the input and generate a response based on the content of the document.
  
  payload = {
    "model": f"{AZURE_AI_DEPLOYMENT_NAME}",
    "messages": [
      {
        "role": "user",
        "content": [
          {
            "type": "text",
            "text": question
          },
        ]
      }
    ],
    "tools": [
    {
      "type": "function",
      "function": {
        "name": "get_weather",
        "description": "Get current temperature for a given location.",
        "parameters": {
          "type": "object",
          "properties": {
            "location": {
              "type": "string",
              "description": "City and country e.g. Bogotá, Colombia"
            }
          },
          "required": [
            "location"
          ],
          "additionalProperties": False
        },
      }
    }
  ]
  }
  return payload


def get_weather(location: str) -> str:
    # This is a mock implementation of the get_weather function. In a real implementation, this function would call a weather API to get the current temperature for the given location. For the purpose of this example, we will return a hardcoded response.
    return f"The current temperature in {location} is 25°C."

In [ ]:
# Registry maps tool names to actual functions
tool_registry = {
    "get_weather": get_weather,
}

payload = documentPayload_tool("What is the weather like in Paris today?")

response = requests.post(
        url=AZURE_MISTRAL_MODEL_ENDPOINT, 
        json=payload,
        headers=REQUEST_HEADERS,
    )
tool_call = response.json()["choices"][0]["message"]["tool_calls"][0]
args = json.loads(tool_call["function"]["arguments"])

# Actually call the local function
# Dispatch via tool_call — no explicit function name needed
func = tool_registry[tool_call["function"]["name"]]
result = func(**args)

print(result)

The current temperature in Paris, France is 25°C.


Tool Calling using MCP server hosted in Azure Foundry: https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/model-context-protocol?pivots=python

Create an Agent in Foundry first.

In [17]:
# Create clients to call Foundry API
project = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT+PROJECT_API_NAME,
    credential=DefaultAzureCredential(), 
)
openai = project.get_openai_client()

# [START tool_declaration]
tool = MCPTool(
    server_label="api-specs",
    server_url=MCP_SERVER_URL,
    require_approval="never",
    project_connection_id=MCP_CONNECTION_NAME,
)
# [END tool_declaration]

# Create a prompt agent with MCP tool capabilities
agent = project.agents.create_version(
    agent_name="MistralAgentMCPTool",
    definition=PromptAgentDefinition(
        model=AZURE_AI_DEPLOYMENT_NAME,
        instructions="Use MCP tools as needed",
        tools=[tool],
    ),
)
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

Agent created (id: MistralAgentMCPTool:1, name: MistralAgentMCPTool, version: 1)


In [18]:
# Create a conversation to maintain context across multiple interactions
conversation = openai.conversations.create()
print(f"Created conversation (id: {conversation.id})")

# Send initial request that will trigger the MCP tool
response = openai.responses.create(
    conversation=conversation.id,
    input="What is my username in my GitHub profile?",
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

# Process any MCP approval requests that were generated
input_list: ResponseInputParam = []
for item in response.output:
    if item.type == "mcp_approval_request" and item.id:
        print("MCP approval requested")
        print(f"  Server: {item.server_label}")
        print(f"  Tool: {getattr(item, 'name', '<unknown>')}")
        print(
            f"  Arguments: {json.dumps(getattr(item, 'arguments', None), indent=2, default=str)}"
        )

        # Approve only after you review the tool call.
        # In production, implement your own approval UX and policy.
        should_approve = (
            input("Approve this MCP tool call? (y/N): ").strip().lower() == "y"
        )
        input_list.append(
            McpApprovalResponse(
                type="mcp_approval_response",
                approve=should_approve,
                approval_request_id=item.id,
            )
        )

# Send the approval response back to continue the agent's work
if input_list:
    response = openai.responses.create(
        input=input_list,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )

print(f"Response: {response.output_text}")

Created conversation (id: conv_3b8249f90056cced00aXUdK600C7kKtFhyZJJ7AikMmfDnpw9C)
Response: Your GitHub username is **`peymanmohajerian`**.


In [7]:
# Clean up resources by deleting the agent version
project.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("Agent deleted")

Agent deleted
